# 01 EDA Template

목표: 데이터 구조, target 분포, 결측치, 이상치, feature redundancy, 데이터 누수 후보를 확인하고 `reports/DATA_CARD.md`에 옮길 근거를 만듭니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/raw/dataset.csv')
TARGET = 'label'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'raw 파일을 찾을 수 없습니다: {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
if TARGET not in df.columns:
    raise ValueError(f'target column이 없습니다: {TARGET}')

df.head()

## 1. 기본 구조

In [ ]:
display(df.shape)
display(df.info())
display(df.describe(include='all').T.head(30))

## 2. 결측치와 중복

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
display(missing[missing > 0])
print('duplicated rows:', df.duplicated().sum())

## 3. Target 분포

In [ ]:
display(df[TARGET].value_counts(dropna=False))
sns.countplot(data=df, x=TARGET)
plt.title('Target distribution')
plt.show()

## 4. Feature 분포, redundancy, 누수 후보

- constant / near-constant 컬럼이 있는가?
- 결측률이 80% 이상인 컬럼이 있는가?
- ID-like 또는 high-cardinality 컬럼이 있는가?
- numeric feature 간 절대 상관계수 0.95 이상인 pair가 있는가?
- target을 직접 계산한 컬럼이 있는가?
- 시간 순서상 미래 정보를 담은 컬럼이 있는가?
- ID, 이름, 결과 후 생성되는 상태값이 feature에 섞였는가?

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.drop(TARGET, errors='ignore')
numeric_df = df[numeric_cols]

if len(numeric_cols) > 0:
    df[numeric_cols[:12]].hist(figsize=(14, 10), bins=30)
    plt.tight_layout()
    plt.show()
else:
    print('numeric feature가 없습니다.')

CORRELATION_THRESHOLD = 0.95
if len(numeric_cols) >= 2:
    corr_matrix = numeric_df.corr().abs()
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, cmap='coolwarm', vmin=0, vmax=1)
    plt.title('Numeric feature correlation matrix (absolute)')
    plt.show()

    upper_mask = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    high_corr_pairs = (
        corr_matrix.where(upper_mask)
        .stack()
        .sort_values(ascending=False)
        .reset_index(name='abs_correlation')
        .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2'})
    )
    display(high_corr_pairs[high_corr_pairs['abs_correlation'] >= CORRELATION_THRESHOLD])
else:
    high_corr_pairs = pd.DataFrame(columns=['feature_1', 'feature_2', 'abs_correlation'])
    print('correlation pair를 계산할 numeric feature가 부족합니다.')

if pd.api.types.is_numeric_dtype(df[TARGET]) and len(numeric_cols) > 0:
    corr_with_target = numeric_df.corrwith(df[TARGET]).abs().sort_values(ascending=False)
    display(corr_with_target.head(20))
else:
    print('target correlation은 numeric target일 때만 계산합니다.')

missing_rate = df.isna().mean()
nunique = df.nunique(dropna=True)
dominant_rate = df.apply(lambda s: s.value_counts(dropna=False, normalize=True).iloc[0] if len(s) else 0)
feature_cols = [col for col in df.columns if col != TARGET]
drop_candidate_rows = []

for col in feature_cols:
    reasons = []
    evidence = []
    if nunique[col] <= 1:
        reasons.append('constant')
        evidence.append(f'nunique={nunique[col]}')
    if dominant_rate[col] >= 0.95 and nunique[col] > 1:
        reasons.append('near_constant')
        evidence.append(f'dominant_rate={dominant_rate[col]:.2%}')
    if missing_rate[col] >= 0.80:
        reasons.append('high_missing')
        evidence.append(f'missing_rate={missing_rate[col]:.2%}')
    if nunique[col] / max(len(df), 1) >= 0.95:
        reasons.append('id_like_high_cardinality')
        evidence.append(f'unique_ratio={nunique[col] / max(len(df), 1):.2%}')
    if reasons:
        drop_candidate_rows.append({
            'column': col,
            'reason': ', '.join(reasons),
            'evidence': '; '.join(evidence),
            'recommended_action': 'review before --drop-columns',
        })

for _, row in high_corr_pairs[high_corr_pairs['abs_correlation'] >= CORRELATION_THRESHOLD].iterrows():
    drop_candidate_rows.append({
        'column': f"{row['feature_1']} / {row['feature_2']}",
        'reason': 'high_correlation_pair',
        'evidence': f"abs_correlation={row['abs_correlation']:.3f}",
        'recommended_action': 'keep one with clearer meaning, fewer missing values, and no leakage risk',
    })

drop_candidates = pd.DataFrame(
    drop_candidate_rows,
    columns=['column', 'reason', 'evidence', 'recommended_action'],
)
display(drop_candidates)

## 5. Split 점검 메모

- stratify가 필요한가?
- 같은 사용자/상품/문서가 train과 test에 동시에 들어가면 안 되는가?
- 시간 기반 split이 더 적절한가?

이 메모를 `reports/DATA_CARD.md`와 `data_manifest.json`에 반영하세요.